In [1]:
# ==============================================================================
# PHASE 1: MOUNT DRIVE & PREPARE WORKSPACE
# ==============================================================================

# 1. Mount Google Drive to access the zipped dataset and save progress
from google.colab import drive
drive.mount('/content/drive')

# 2. Install Ultralytics quietly
!pip install ultralytics -q

# 3. Copy the zipped dataset from Drive to Colab's fast local storage
!cp "/content/drive/MyDrive/RoadEye_Workspace_V6/RoadEye_V6_Dataset.zip" "/content/"

# 4. Unzip the dataset locally and silently
!unzip -q /content/RoadEye_V6_Dataset.zip -d /content/dataset_v6

print("✅ Environment setup complete. Data is unzipped locally on Colab.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.7 MB/s eta 0:00:00
✅ Environment setup complete. Data is unzipped locally on Colab.


In [2]:
# ==============================================================================
# PHASE 2: GENERATE YAML CONFIGURATION
# ==============================================================================
import yaml

yaml_content = {
    "path": "/content/dataset_v6",
    "train": "RoadEye_V6_Dataset/train/images",
    "val": "RoadEye_V6_Dataset/valid/images",
    "test": "RoadEye_V6_Dataset/test/images",
    "names": {
        0: "Crack",
        1: "Pothole"
    }
}

yaml_path = "/content/roadeye_v6.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ YAML file generated successfully at {yaml_path}")

✅ YAML file generated successfully at /content/roadeye_v6.yaml


In [3]:
# # ==============================================================================
# # PHASE 3: YOLOv8m TRAINING EXECUTION
# # ==============================================================================
# from ultralytics import YOLO

# # 1. Initialize the medium model from scratch
# model = YOLO("yolov8m.pt")

# # 2. Execute training with the agreed-upon strict parameters
# results = model.train(
#     data="/content/roadeye_v6.yaml",

#     # --------------------------------------------------
#     # Project & Save Directories (Directly to Drive)
#     # --------------------------------------------------
#     project="/content/drive/MyDrive/RoadEye_Workspace_V6/Runs",
#     name="V6_Accuracy_Track",
#     save=True,
#     save_period=5,

#     # --------------------------------------------------
#     # Core Architecture Limits
#     # --------------------------------------------------
#     epochs=150,
#     imgsz=1024,
#     batch=4,                 # Reduced to 4 to prevent OOM on Colab Free Tier (T4)
#     device=0,

#     # --------------------------------------------------
#     # Optimizer & Learning Rate
#     # --------------------------------------------------
#     optimizer="SGD",
#     lr0=0.008,
#     lrf=0.12,
#     cos_lr=True,

#     # --------------------------------------------------
#     # Loss Function Weights
#     # --------------------------------------------------
#     box=7.5,
#     cls=0.7,
#     dfl=1.5,

#     # --------------------------------------------------
#     # Augmentations (Clean Domain Strategy)
#     # --------------------------------------------------
#     mosaic=0.85,
#     close_mosaic=35,
#     mixup=0.0,
#     copy_paste=0.0,

#     # --------------------------------------------------
#     # Callbacks & Memory Management
#     # --------------------------------------------------
#     patience=40,
#     amp=True,
#     seed=42
# )

# print("🏆 Training initiated! All checkpoints are being saved directly to your Google Drive.")

In [6]:
import os
import cv2
import math
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import shutil

# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================
MODEL_PATH = "/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6_Accuracy_Track-3/weights/best.pt"
TEST_IMAGES_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/images")
TEST_LABELS_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/labels")

GALLERY_DIR = Path("/content/RoadEye_Error_Gallery")
FP_DIR = GALLERY_DIR / "False_Positives"
FN_DIR = GALLERY_DIR / "False_Negatives"

IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.25
MAX_GALLERY_IMAGES = 20

# Reset Gallery Directories
if GALLERY_DIR.exists(): shutil.rmtree(GALLERY_DIR)
FP_DIR.mkdir(parents=True, exist_ok=True)
FN_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# METRICS TRACKER
# ==========================================================
stats = {
    "class": {0: {"TP": 0, "FP": 0, "FN": 0}, 1: {"TP": 0, "FP": 0, "FN": 0}},
    "country": {"Japan": {"TP": 0, "FP": 0, "FN": 0}, "India": {"TP": 0, "FP": 0, "FN": 0}},
    "country_class": {
        "Japan": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}},
        "India": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}}
    },
    "size": {"16-32": {"TP": 0, "FN": 0}, "32-64": {"TP": 0, "FN": 0},
             "64-128": {"TP": 0, "FN": 0}, "128+": {"TP": 0, "FN": 0}},
    "density": {"1": {"TP": 0, "FN": 0}, "2": {"TP": 0, "FN": 0},
                "3-5": {"TP": 0, "FN": 0}, "6+": {"TP": 0, "FN": 0}},
    "aspect": {"tall": {"TP": 0, "FN": 0}, "square": {"TP": 0, "FN": 0}, "wide": {"TP": 0, "FN": 0}},
    "conf_dist": {"0.25-0.4": 0, "0.4-0.6": 0, "0.6-0.8": 0, "0.8-1.0": 0}
}

# Lists for Error Gallery
fp_gallery = [] # (confidence, img_path, p_box, cls_id)
fn_gallery = [] # (size_area, img_path, g_box, cls_id)

def box_iou(box1, box2):
    """Calculate IoU between two bounding boxes [cx, cy, w, h] (normalized)."""
    x1_1, y1_1 = box1[0] - box1[2]/2, box1[1] - box1[3]/2
    x2_1, y2_1 = box1[0] + box1[2]/2, box1[1] + box1[3]/2
    x1_2, y1_2 = box2[0] - box2[2]/2, box2[1] - box2[3]/2
    x2_2, y2_2 = box2[0] + box2[2]/2, box2[1] + box2[3]/2
    xi1, yi1 = max(x1_1, x1_2), max(y1_1, y1_2)
    xi2, yi2 = min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union_area = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter_area
    return inter_area / union_area if union_area > 0 else 0

def get_geom_attrs(w_norm, h_norm, img_w, img_h):
    size = math.sqrt((w_norm * img_w) * (h_norm * img_h))
    if size < 32: s_bin = "16-32"
    elif size < 64: s_bin = "32-64"
    elif size < 128: s_bin = "64-128"
    else: s_bin = "128+"

    ratio = w_norm / h_norm if h_norm > 0 else 1
    if ratio > 1.5: a_bin = "wide"
    elif ratio < 0.67: a_bin = "tall"
    else: a_bin = "square"
    return s_bin, a_bin, size

def calc_metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p * 100, r * 100, f1 * 100

def draw_and_save_error(img_path, box_norm, cls_id, save_dir, prefix, metric_val, is_fp=True):
    img = cv2.imread(str(img_path))
    if img is None: return
    h, w = img.shape[:2]

    cx, cy, bw, bh = box_norm
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    x2 = int((cx + bw/2) * w)
    y2 = int((cy + bh/2) * h)

    color = (255, 0, 0) if is_fp else (0, 0, 255) # Blue for FP, Red for FN (BGR)
    label = f"FP {cls_id} (Conf: {metric_val:.2f})" if is_fp else f"FN {cls_id} (Size: {metric_val:.0f}px)"

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    cv2.putText(img, label, (x1, max(10, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    save_name = f"{prefix}_{img_path.name}"
    cv2.imwrite(str(save_dir / save_name), img)

def run_forensics():
    print("🚀 Loading Model & Starting Pro Forensic Analysis...")
    model = YOLO(MODEL_PATH)
    image_paths = list(TEST_IMAGES_DIR.glob("*.*"))

    for idx, img_path in enumerate(image_paths):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']: continue

        lbl_path = TEST_LABELS_DIR / f"{img_path.stem}.txt"
        gt_boxes = {0: [], 1: []}

        img = cv2.imread(str(img_path))
        if img is None: continue
        img_h, img_w = img.shape[:2]

        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) >= 5: gt_boxes[int(parts[0])].append(parts[1:5])

        total_gt = len(gt_boxes[0]) + len(gt_boxes[1])
        density_bin = "1" if total_gt == 1 else "2" if total_gt == 2 else "3-5" if 3 <= total_gt <= 5 else "6+"
        country = "Japan" if img_path.name.startswith("jp_") else "India"

        results = model.predict(source=str(img_path), imgsz=1024, conf=CONF_THRESHOLD, verbose=False)
        pred_data = results[0].boxes
        pred_boxes = {0: [], 1: []}

        if pred_data is not None and len(pred_data) > 0:
            boxes_norm = pred_data.xywhn.cpu().numpy()
            classes = pred_data.cls.cpu().numpy()
            confs = pred_data.conf.cpu().numpy()
            for i in range(len(classes)):
                pred_boxes[int(classes[i])].append((boxes_norm[i], confs[i]))
                c_val = confs[i]
                if 0.25 <= c_val < 0.4: stats["conf_dist"]["0.25-0.4"] += 1
                elif 0.4 <= c_val < 0.6: stats["conf_dist"]["0.4-0.6"] += 1
                elif 0.6 <= c_val < 0.8: stats["conf_dist"]["0.6-0.8"] += 1
                else: stats["conf_dist"]["0.8-1.0"] += 1

        for cls_id in [0, 1]:
            gts = gt_boxes[cls_id]
            preds = sorted(pred_boxes[cls_id], key=lambda x: x[1], reverse=True)
            matched_gt = set()

            for p_box, conf in preds:
                best_iou = 0
                best_gt_idx = -1
                for gt_idx, g_box in enumerate(gts):
                    if gt_idx in matched_gt: continue
                    iou = box_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou, best_gt_idx = iou, gt_idx

                if best_iou >= IOU_THRESHOLD:
                    matched_gt.add(best_gt_idx)
                    stats["class"][cls_id]["TP"] += 1
                    stats["country"][country]["TP"] += 1
                    stats["country_class"][country][cls_id]["TP"] += 1

                    s_bin, a_bin, _ = get_geom_attrs(gts[best_gt_idx][2], gts[best_gt_idx][3], img_w, img_h)
                    stats["size"][s_bin]["TP"] += 1
                    stats["aspect"][a_bin]["TP"] += 1
                    if total_gt > 0: stats["density"][density_bin]["TP"] += 1
                else:
                    stats["class"][cls_id]["FP"] += 1
                    stats["country"][country]["FP"] += 1
                    stats["country_class"][country][cls_id]["FP"] += 1
                    fp_gallery.append((conf, img_path, p_box, cls_id))

            for gt_idx, g_box in enumerate(gts):
                if gt_idx not in matched_gt:
                    stats["class"][cls_id]["FN"] += 1
                    stats["country"][country]["FN"] += 1
                    stats["country_class"][country][cls_id]["FN"] += 1

                    s_bin, a_bin, raw_size = get_geom_attrs(g_box[2], g_box[3], img_w, img_h)
                    stats["size"][s_bin]["FN"] += 1
                    stats["aspect"][a_bin]["FN"] += 1
                    if total_gt > 0: stats["density"][density_bin]["FN"] += 1
                    fn_gallery.append((raw_size, img_path, g_box, cls_id))

        if idx % 100 == 0: print(f"   -> Analyzed {idx}/{len(image_paths)} images...")

    print("\n📸 Generating Error Gallery...")
    fp_gallery.sort(key=lambda x: x[0], reverse=True) # Top Confident Mistakes
    fn_gallery.sort(key=lambda x: x[0], reverse=True) # Biggest Missed Defects

    for i, (conf, p, b, c) in enumerate(fp_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FP_DIR, f"top{i+1}_fp", conf, is_fp=True)
    for i, (sz, p, b, c) in enumerate(fn_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FN_DIR, f"top{i+1}_fn", sz, is_fp=False)

    print_report()

def print_report():
    print("\n" + "="*70)
    print(" 🚨 ROADEYE V6 ULTIMATE FORENSIC REPORT 🚨")
    print("="*70)

    print("\n[ 1. Class Analysis ]")
    print(f"{'Category':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for name, cid in [("Crack", 0), ("Pothole", 1)]:
        s = stats["class"][cid]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{name:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 2. Country Analysis ]")
    print(f"{'Country':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        s = stats["country"][c]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{c:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 3. Per-Country Per-Class Analysis (⭐⭐⭐⭐⭐) ]")
    print(f"{'Country':<8} | {'Class':<8} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'Prec.':<6} | {'Recall':<6} | {'F1':<6}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        for name, cid in [("Crack", 0), ("Pothole", 1)]:
            s = stats["country_class"][c][cid]
            p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
            print(f"{c:<8} | {name:<8} | {s['TP']:<4} | {s['FP']:<4} | {s['FN']:<4} | {p:>5.1f}% | {r:>5.1f}% | {f1:>5.1f}%")

    print("\n[ 4. Confidence Distribution (Model Certainty) ]")
    for b, c in stats["conf_dist"].items(): print(f"   Conf {b:<8}: {c} predictions")

    print("\n[ 5. Geometric Recalls (Size & Aspect Ratio) ]")
    print("   [Size]")
    for s in ["16-32", "32-64", "64-128", "128+"]:
        r = calc_metrics(stats["size"][s]["TP"], 0, stats["size"][s]["FN"])[1]
        print(f"      {s:<8}: {r:.1f}%")
    print("   [Aspect Ratio]")
    for a in ["tall", "square", "wide"]:
        r = calc_metrics(stats["aspect"][a]["TP"], 0, stats["aspect"][a]["FN"])[1]
        print(f"      {a:<8}: {r:.1f}%")

    print("\n[ 6. Density Recall ]")
    for d in ["1", "2", "3-5", "6+"]:
        r = calc_metrics(stats["density"][d]["TP"], 0, stats["density"][d]["FN"])[1]
        print(f"   {d:<8} objects : {r:.1f}%")

    print("\n📸 Error Gallery Exported! Check '/content/RoadEye_Error_Gallery'")
    print("="*70)

if __name__ == "__main__":
    run_forensics()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Loading Model & Starting Pro Forensic Analysis...
   -> Analyzed 0/746 images...
   -> Analyzed 100/746 images...
   -> Analyzed 200/746 images...
   -> Analyzed 300/746 images...
   -> Analyzed 400/746 images...
   -> Analyzed 500/746 images...
   -> Analyzed 600/746 images...
   -> Analyzed 700/746 images...

📸 Generating Error Gallery...

 🚨 ROADEYE V6 ULTIMATE FORENSIC REPORT 🚨

[ 1. Class Analysis ]
Category   | TP    | FP    | FN    | Prec.   | Recall  | F1-Score
----------------------------------------------------------------------
Crack      | 583   | 444   | 554   |   56.8% |   51.3% |   53.9%
Pothole    | 256   | 174   | 215   |   59.5% |   54.4% |   56.8%

[ 2. Countr

In [5]:
import os
import cv2
import math
from pathlib import Path

# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================
TEST_IMAGES_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/images")
TEST_LABELS_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/labels")

# Metrics Tracker
stats = {
    "Japan": {"<16": 0, "16-32": 0, "32-64": 0, ">64": 0},
    "India": {"<16": 0, "16-32": 0, "32-64": 0, ">64": 0}
}

def check_test_set_leakage():
    print("🔍 Inspecting Test Set for Microscopic Objects (<16px)...\n")

    image_paths = list(TEST_IMAGES_DIR.glob("*.*"))

    for img_path in image_paths:
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue

        lbl_path = TEST_LABELS_DIR / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        country = "Japan" if img_path.name.startswith("jp_") else "India"

        # Read image to get actual dimensions for accurate pixel calculation
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    w_norm, h_norm = float(parts[3]), float(parts[4])

                    # Calculate geometric mean size in actual pixels
                    px_w = w_norm * img_w
                    px_h = h_norm * img_h
                    size = math.sqrt(px_w * px_h)

                    if size < 16:
                        stats[country]["<16"] += 1
                    elif size < 32:
                        stats[country]["16-32"] += 1
                    elif size < 64:
                        stats[country]["32-64"] += 1
                    else:
                        stats[country][">64"] += 1

    # Print Report
    print("=========================================")
    print(" 📏 TEST SET BOUNDING BOX SIZE ANALYSIS")
    print("=========================================")
    print(f"{'Country':<10} | {'< 16 px':<10} | {'16-32 px':<10} | {'32-64 px':<10} | {'> 64 px':<10}")
    print("-" * 65)

    for c in ["Japan", "India"]:
        r_16 = stats[c]["<16"]
        r_32 = stats[c]["16-32"]
        r_64 = stats[c]["32-64"]
        r_plus = stats[c][">64"]
        print(f"{c:<10} | {r_16:<10} | {r_32:<10} | {r_64:<10} | {r_plus:<10}")

    print("=========================================")

if __name__ == "__main__":
    check_test_set_leakage()

🔍 Inspecting Test Set for Microscopic Objects (<16px)...

 📏 TEST SET BOUNDING BOX SIZE ANALYSIS
Country    | < 16 px    | 16-32 px   | 32-64 px   | > 64 px   
-----------------------------------------------------------------
Japan      | 0          | 100        | 428        | 820       
India      | 0          | 16         | 67         | 177       


In [7]:
import shutil
from google.colab import files

# ==========================================================
# ZIP AND DOWNLOAD TO LOCAL MACHINE
# ==========================================================

# 1. Archive the entire gallery directory into a zip file
shutil.make_archive("/content/RoadEye_Error_Gallery", 'zip', "/content/RoadEye_Error_Gallery")

# 2. Trigger the browser to download the zip file
files.download("/content/RoadEye_Error_Gallery.zip")

print("✅ Download triggered! Check your browser's downloads.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered! Check your browser's downloads.


In [8]:
# ==============================================================================
# ROADEYE V6.1 - DOMAIN ADAPTATION (FINE-TUNING PHASE)
# ==============================================================================
from ultralytics import YOLO

# 1. Load the best weights from V6 (Epoch 60)
model = YOLO("/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6_Accuracy_Track-3/weights/best.pt")

# 2. Execute fine-tuning with controlled augmentations
results = model.train(
    data="/content/roadeye_v6.yaml",

    # Project Paths
    project="/content/drive/MyDrive/RoadEye_Workspace_V6/Runs",
    name="V6.1_Controlled_FineTuning",
    save=True,
    save_period=5,

    # Core Architecture (Shorter run for fine-tuning)
    epochs=60,
    imgsz=1024,
    batch=4,
    device=0,

    # Optimizer (Lower Learning Rate to prevent catastrophic forgetting)
    optimizer="SGD",
    lr0=0.002,
    lrf=0.12,
    cos_lr=True,

    # Loss Weights
    box=7.5,
    cls=0.7,
    dfl=1.5,

    # Color Domain Adaptation (India Fix)
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,

    # Spatial Robustness
    translate=0.15,
    scale=0.35,
    fliplr=0.5,
    flipud=0.0,

    # Crowd Robustness
    mosaic=0.85,
    close_mosaic=15,

    # Light Texture Blending
    mixup=0.05,
    copy_paste=0.0,

    # Training Control
    patience=20,
    amp=True,
    seed=42
)

print("🏆 RoadEye V6.1 Fine-Tuning started on V6 best weights.")

Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/roadeye_v6.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.12, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6_Accuracy_Track-3/weights/best.pt, momentum=0.937, mosaic=0.85, multi_scale=0.0, name=V6.1_Controlled_FineTuning, nbs=64,

In [9]:
import os
import cv2
import math
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import shutil

# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================
# 🔴 UPDATED: Pointing to the new V6.1 Fine-Tuned Model
MODEL_PATH = "/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1_Controlled_FineTuning/weights/best.pt"
TEST_IMAGES_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/images")
TEST_LABELS_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/labels")

GALLERY_DIR = Path("/content/RoadEye_Error_Gallery_V6_1")
FP_DIR = GALLERY_DIR / "False_Positives"
FN_DIR = GALLERY_DIR / "False_Negatives"

IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.25
MAX_GALLERY_IMAGES = 20

# Reset Gallery Directories for V6.1
if GALLERY_DIR.exists(): shutil.rmtree(GALLERY_DIR)
FP_DIR.mkdir(parents=True, exist_ok=True)
FN_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# METRICS TRACKER
# ==========================================================
stats = {
    "class": {0: {"TP": 0, "FP": 0, "FN": 0}, 1: {"TP": 0, "FP": 0, "FN": 0}},
    "country": {"Japan": {"TP": 0, "FP": 0, "FN": 0}, "India": {"TP": 0, "FP": 0, "FN": 0}},
    "country_class": {
        "Japan": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}},
        "India": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}}
    },
    "size": {"16-32": {"TP": 0, "FN": 0}, "32-64": {"TP": 0, "FN": 0},
             "64-128": {"TP": 0, "FN": 0}, "128+": {"TP": 0, "FN": 0}},
    "density": {"1": {"TP": 0, "FN": 0}, "2": {"TP": 0, "FN": 0},
                "3-5": {"TP": 0, "FN": 0}, "6+": {"TP": 0, "FN": 0}},
    "aspect": {"tall": {"TP": 0, "FN": 0}, "square": {"TP": 0, "FN": 0}, "wide": {"TP": 0, "FN": 0}},
    "conf_dist": {"0.25-0.4": 0, "0.4-0.6": 0, "0.6-0.8": 0, "0.8-1.0": 0}
}

# Lists for Error Gallery
fp_gallery = [] # (confidence, img_path, p_box, cls_id)
fn_gallery = [] # (size_area, img_path, g_box, cls_id)

def box_iou(box1, box2):
    """Calculate IoU between two bounding boxes [cx, cy, w, h] (normalized)."""
    x1_1, y1_1 = box1[0] - box1[2]/2, box1[1] - box1[3]/2
    x2_1, y2_1 = box1[0] + box1[2]/2, box1[1] + box1[3]/2
    x1_2, y1_2 = box2[0] - box2[2]/2, box2[1] - box2[3]/2
    x2_2, y2_2 = box2[0] + box2[2]/2, box2[1] + box2[3]/2
    xi1, yi1 = max(x1_1, x1_2), max(y1_1, y1_2)
    xi2, yi2 = min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union_area = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter_area
    return inter_area / union_area if union_area > 0 else 0

def get_geom_attrs(w_norm, h_norm, img_w, img_h):
    size = math.sqrt((w_norm * img_w) * (h_norm * img_h))
    if size < 32: s_bin = "16-32"
    elif size < 64: s_bin = "32-64"
    elif size < 128: s_bin = "64-128"
    else: s_bin = "128+"

    ratio = w_norm / h_norm if h_norm > 0 else 1
    if ratio > 1.5: a_bin = "wide"
    elif ratio < 0.67: a_bin = "tall"
    else: a_bin = "square"
    return s_bin, a_bin, size

def calc_metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p * 100, r * 100, f1 * 100

def draw_and_save_error(img_path, box_norm, cls_id, save_dir, prefix, metric_val, is_fp=True):
    img = cv2.imread(str(img_path))
    if img is None: return
    h, w = img.shape[:2]

    cx, cy, bw, bh = box_norm
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    x2 = int((cx + bw/2) * w)
    y2 = int((cy + bh/2) * h)

    color = (255, 0, 0) if is_fp else (0, 0, 255) # Blue for FP, Red for FN (BGR)
    label = f"FP {cls_id} (Conf: {metric_val:.2f})" if is_fp else f"FN {cls_id} (Size: {metric_val:.0f}px)"

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    cv2.putText(img, label, (x1, max(10, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    save_name = f"{prefix}_{img_path.name}"
    cv2.imwrite(str(save_dir / save_name), img)

def run_forensics():
    print("🚀 Loading V6.1 Model & Starting Pro Forensic Analysis...")
    model = YOLO(MODEL_PATH)
    image_paths = list(TEST_IMAGES_DIR.glob("*.*"))

    for idx, img_path in enumerate(image_paths):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']: continue

        lbl_path = TEST_LABELS_DIR / f"{img_path.stem}.txt"
        gt_boxes = {0: [], 1: []}

        img = cv2.imread(str(img_path))
        if img is None: continue
        img_h, img_w = img.shape[:2]

        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) >= 5: gt_boxes[int(parts[0])].append(parts[1:5])

        total_gt = len(gt_boxes[0]) + len(gt_boxes[1])
        density_bin = "1" if total_gt == 1 else "2" if total_gt == 2 else "3-5" if 3 <= total_gt <= 5 else "6+"
        country = "Japan" if img_path.name.startswith("jp_") else "India"

        results = model.predict(source=str(img_path), imgsz=1024, conf=CONF_THRESHOLD, verbose=False)
        pred_data = results[0].boxes
        pred_boxes = {0: [], 1: []}

        if pred_data is not None and len(pred_data) > 0:
            boxes_norm = pred_data.xywhn.cpu().numpy()
            classes = pred_data.cls.cpu().numpy()
            confs = pred_data.conf.cpu().numpy()
            for i in range(len(classes)):
                pred_boxes[int(classes[i])].append((boxes_norm[i], confs[i]))
                c_val = confs[i]
                if 0.25 <= c_val < 0.4: stats["conf_dist"]["0.25-0.4"] += 1
                elif 0.4 <= c_val < 0.6: stats["conf_dist"]["0.4-0.6"] += 1
                elif 0.6 <= c_val < 0.8: stats["conf_dist"]["0.6-0.8"] += 1
                else: stats["conf_dist"]["0.8-1.0"] += 1

        for cls_id in [0, 1]:
            gts = gt_boxes[cls_id]
            preds = sorted(pred_boxes[cls_id], key=lambda x: x[1], reverse=True)
            matched_gt = set()

            for p_box, conf in preds:
                best_iou = 0
                best_gt_idx = -1
                for gt_idx, g_box in enumerate(gts):
                    if gt_idx in matched_gt: continue
                    iou = box_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou, best_gt_idx = iou, gt_idx

                if best_iou >= IOU_THRESHOLD:
                    matched_gt.add(best_gt_idx)
                    stats["class"][cls_id]["TP"] += 1
                    stats["country"][country]["TP"] += 1
                    stats["country_class"][country][cls_id]["TP"] += 1

                    s_bin, a_bin, _ = get_geom_attrs(gts[best_gt_idx][2], gts[best_gt_idx][3], img_w, img_h)
                    stats["size"][s_bin]["TP"] += 1
                    stats["aspect"][a_bin]["TP"] += 1
                    if total_gt > 0: stats["density"][density_bin]["TP"] += 1
                else:
                    stats["class"][cls_id]["FP"] += 1
                    stats["country"][country]["FP"] += 1
                    stats["country_class"][country][cls_id]["FP"] += 1
                    fp_gallery.append((conf, img_path, p_box, cls_id))

            for gt_idx, g_box in enumerate(gts):
                if gt_idx not in matched_gt:
                    stats["class"][cls_id]["FN"] += 1
                    stats["country"][country]["FN"] += 1
                    stats["country_class"][country][cls_id]["FN"] += 1

                    s_bin, a_bin, raw_size = get_geom_attrs(g_box[2], g_box[3], img_w, img_h)
                    stats["size"][s_bin]["FN"] += 1
                    stats["aspect"][a_bin]["FN"] += 1
                    if total_gt > 0: stats["density"][density_bin]["FN"] += 1
                    fn_gallery.append((raw_size, img_path, g_box, cls_id))

        if idx % 100 == 0: print(f"   -> Analyzed {idx}/{len(image_paths)} images...")

    print("\n📸 Generating Error Gallery for V6.1...")
    fp_gallery.sort(key=lambda x: x[0], reverse=True) # Top Confident Mistakes
    fn_gallery.sort(key=lambda x: x[0], reverse=True) # Biggest Missed Defects

    for i, (conf, p, b, c) in enumerate(fp_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FP_DIR, f"top{i+1}_fp", conf, is_fp=True)
    for i, (sz, p, b, c) in enumerate(fn_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FN_DIR, f"top{i+1}_fn", sz, is_fp=False)

    print_report()

def print_report():
    print("\n" + "="*70)
    print(" 🚨 ROADEYE V6.1 ULTIMATE FORENSIC REPORT 🚨")
    print("="*70)

    print("\n[ 1. Class Analysis ]")
    print(f"{'Category':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for name, cid in [("Crack", 0), ("Pothole", 1)]:
        s = stats["class"][cid]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{name:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 2. Country Analysis ]")
    print(f"{'Country':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        s = stats["country"][c]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{c:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 3. Per-Country Per-Class Analysis (⭐⭐⭐⭐⭐) ]")
    print(f"{'Country':<8} | {'Class':<8} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'Prec.':<6} | {'Recall':<6} | {'F1':<6}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        for name, cid in [("Crack", 0), ("Pothole", 1)]:
            s = stats["country_class"][c][cid]
            p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
            print(f"{c:<8} | {name:<8} | {s['TP']:<4} | {s['FP']:<4} | {s['FN']:<4} | {p:>5.1f}% | {r:>5.1f}% | {f1:>5.1f}%")

    print("\n[ 4. Confidence Distribution (Model Certainty) ]")
    for b, c in stats["conf_dist"].items(): print(f"   Conf {b:<8}: {c} predictions")

    print("\n[ 5. Geometric Recalls (Size & Aspect Ratio) ]")
    print("   [Size]")
    for s in ["16-32", "32-64", "64-128", "128+"]:
        r = calc_metrics(stats["size"][s]["TP"], 0, stats["size"][s]["FN"])[1]
        print(f"      {s:<8}: {r:.1f}%")
    print("   [Aspect Ratio]")
    for a in ["tall", "square", "wide"]:
        r = calc_metrics(stats["aspect"][a]["TP"], 0, stats["aspect"][a]["FN"])[1]
        print(f"      {a:<8}: {r:.1f}%")

    print("\n[ 6. Density Recall ]")
    for d in ["1", "2", "3-5", "6+"]:
        r = calc_metrics(stats["density"][d]["TP"], 0, stats["density"][d]["FN"])[1]
        print(f"   {d:<8} objects : {r:.1f}%")

    print("\n📸 Error Gallery Exported! Check '/content/RoadEye_Error_Gallery_V6_1'")
    print("="*70)

if __name__ == "__main__":
    run_forensics()

🚀 Loading V6.1 Model & Starting Pro Forensic Analysis...
   -> Analyzed 0/746 images...
   -> Analyzed 100/746 images...
   -> Analyzed 200/746 images...
   -> Analyzed 300/746 images...
   -> Analyzed 400/746 images...
   -> Analyzed 500/746 images...
   -> Analyzed 600/746 images...
   -> Analyzed 700/746 images...

📸 Generating Error Gallery for V6.1...

 🚨 ROADEYE V6.1 ULTIMATE FORENSIC REPORT 🚨

[ 1. Class Analysis ]
Category   | TP    | FP    | FN    | Prec.   | Recall  | F1-Score
----------------------------------------------------------------------
Crack      | 590   | 437   | 547   |   57.4% |   51.9% |   54.5%
Pothole    | 296   | 186   | 175   |   61.4% |   62.8% |   62.1%

[ 2. Country Analysis ]
Country    | TP    | FP    | FN    | Prec.   | Recall  | F1-Score
----------------------------------------------------------------------
Japan      | 757   | 526   | 591   |   59.0% |   56.2% |   57.5%
India      | 129   | 97    | 131   |   57.1% |   49.6% |   53.1%

[ 3. Per-Count

In [10]:
# ==============================================================================
# OFFICIAL mAP EVALUATION ON THE UNSEEN TEST SET (V6.1)
# ==============================================================================
from ultralytics import YOLO

print("🚀 Loading V6.1 Model for Official Test Set Evaluation...")

# 1. Load the fine-tuned V6.1 model
model = YOLO("/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1_Controlled_FineTuning/weights/best.pt")

# 2. Run the validation engine specifically on the 'test' split
metrics = model.val(
    data="/content/roadeye_v6.yaml",
    split="test",       # 🔴 Crucial: Forces YOLO to use the test set, not validation
    imgsz=1024,
    batch=4,
    device=0,
    plots=True          # Generates PR curves and confusion matrices
)

# 3. Extract and print the exact mAP scores
print("\n=========================================")
print(" 🏆 V6.1 OFFICIAL TEST SET mAP SCORES")
print("=========================================")
print(f"Overall mAP50     : {metrics.box.map50:.4f} ({(metrics.box.map50 * 100):.2f}%)")
print(f"Overall mAP50-95  : {metrics.box.map:.4f} ({(metrics.box.map * 100):.2f}%)")
print("-" * 41)
print(f"Crack mAP50       : {metrics.box.maps[0]:.4f} ({(metrics.box.maps[0] * 100):.2f}%)")
print(f"Pothole mAP50     : {metrics.box.maps[1]:.4f} ({(metrics.box.maps[1] * 100):.2f}%)")
print("=========================================\n")

🚀 Loading V6.1 Model for Official Test Set Evaluation...
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1393.0±483.8 MB/s, size: 72.0 KB)
val: Scanning /content/dataset_v6/RoadEye_V6_Dataset/test/labels... 746 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 746/746 1.3Kit/s 0.6s
val: New cache created: /content/dataset_v6/RoadEye_V6_Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 187/187 3.7it/s 51.1s
                   all        746       1608      0.606      0.572      0.602      0.279
                 Crack        641       1137      0.602      0.529      0.588      0.284
               Pothole        276        471      0.611      0.614      0.615      0.274
Speed: 2.8ms preprocess, 60.2ms inference, 0.0ms loss, 1.0ms postproce

In [ ]:
# ==============================================================================
# ROADEYE V6.1B - CONTROLLED FINE-TUNING & REPRODUCIBILITY
# ==============================================================================
from ultralytics import YOLO

# 1. Initialize model from the best pre-trained V6 weights
model = YOLO("/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6_Accuracy_Track-3/weights/best.pt")

# 2. Execute Controlled Fine-Tuning
results = model.train(
    data="/content/roadeye_v6.yaml",

    # ==================================================
    # Project Paths
    # ==================================================
    project="/content/drive/MyDrive/RoadEye_Workspace_V6/Runs",
    name="V6.1B_Controlled_FineTuning",
    save=True,
    save_period=5,

    # ==================================================
    # Core Architecture & Reproducibility
    # ==================================================
    epochs=50,
    imgsz=1024,
    batch=4,
    device=0,
    workers=2,               # Optimized data loading
    cache=True,              # RAM caching for faster epochs
    deterministic=True,      # Ensures reproducible results
    seed=42,

    # ==================================================
    # Optimizer (Micro-Adjustments)
    # ==================================================
    optimizer="SGD",
    lr0=0.0005,              # Extremely low for fine-tuning stability
    lrf=0.01,                # Aggressive cooldown to prevent forgetting
    cos_lr=True,

    # ==================================================
    # Loss Weights
    # ==================================================
    box=7.5,
    cls=0.7,
    dfl=1.5,

    # ==================================================
    # 🔴 COLOR DOMAIN ADAPTATION (India Fix)
    # ==================================================
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,

    # ==================================================
    # 🔴 SPATIAL ROBUSTNESS (Protected Small Cracks)
    # ==================================================
    translate=0.15,
    scale=0.25,              # Strictly capped to protect 16-32px defects
    fliplr=0.5,
    flipud=0.0,

    # ==================================================
    # 🔴 CROWD ROBUSTNESS (Balanced)
    # ==================================================
    mosaic=0.80,             # Balanced value for crowd vs realism
    close_mosaic=5,          # Shorter cooldown relative to 50 epochs

    # ==================================================
    # 🔴 TEXTURE BLENDING (Sweet Spot)
    # ==================================================
    mixup=0.08,              # Strong enough to blend textures, weak enough to preserve edges
    copy_paste=0.0,          # Strictly disabled

    # ==================================================
    # TRAINING CONTROL
    # ==================================================
    patience=15,             # Gives the low LR enough time to adapt
    amp=True
)

print("🏆 RoadEye V6.1B Controlled Fine-Tuning execution completed.")


Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/roadeye_v6.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.08, mode=train, model=/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6_Accuracy_Track-3/weights/best.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=V6.1B_Controlled_FineTuning, nbs=64, 